[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/sodacore-certified/notebooks/day-05-custom-sql-checks.ipynb#scrollTo=1a2b3c4d)

---
# Day 5 · Custom SQL Checks and Metric Expressions
**certified-journeys / sodacore-certified** · Day 5 · Practice

> **Goal for today:** Write SQL-backed failed-rows checks, define computed custom metrics (like revenue per order), assert referential integrity between tables, and combine warn/fail percentage thresholds — all using SodaCL's `failed rows` and `user-defined metrics` syntax.


In [ ]:
%pip install -q soda-core-duckdb


## Step 1 · Create the Synthetic Schema

Today we need three tables to demonstrate all four check patterns:

| Table | Columns | Purpose |
|---|---|---|
| `orders` | id, customer_id, revenue, order_date, shipped_date | Shipping-date integrity + custom metrics |
| `customers` | id, name | Referential integrity target |
| `order_items` | id, order_id, amount | Failed-rows threshold demo |

We deliberately insert **bad rows** in `orders` (shipped before ordered) and rows in `order_items` that reference non-existent orders to trigger check failures.


In [ ]:
import duckdb
import tempfile
import pathlib

tmpdir = pathlib.Path(tempfile.mkdtemp())
db_path = str(tmpdir / "ecommerce.duckdb")

conn = duckdb.connect(db_path)

# ── customers ────────────────────────────────────────────────────────────────
conn.execute("""
    CREATE TABLE customers (
        id   INTEGER PRIMARY KEY,
        name VARCHAR
    )
""")
conn.execute("""
    INSERT INTO customers VALUES
        (1, 'Alice'), (2, 'Bob'), (3, 'Carol'),
        (4, 'Dave'),  (5, 'Eve')
""")

# ── orders ───────────────────────────────────────────────────────────────────
conn.execute("""
    CREATE TABLE orders (
        id            INTEGER,
        customer_id   INTEGER,
        revenue       DOUBLE,
        order_date    DATE,
        shipped_date  DATE
    )
""")
# Good rows: shipped AFTER order_date
# Bad rows:  shipped BEFORE order_date (business-logic violation)
# Bad row:   customer_id = 999 (no matching customer)
conn.execute("""
    INSERT INTO orders VALUES
        (1,  1, 120.00, DATE '2024-01-10', DATE '2024-01-12'),  -- good
        (2,  2, 340.50, DATE '2024-01-11', DATE '2024-01-15'),  -- good
        (3,  3,  80.00, DATE '2024-01-12', DATE '2024-01-10'),  -- BAD: shipped before ordered
        (4,  4, 500.00, DATE '2024-01-13', DATE '2024-01-14'),  -- good
        (5,  5, 210.00, DATE '2024-01-14', DATE '2024-01-16'),  -- good
        (6, 999,  90.00, DATE '2024-01-15', DATE '2024-01-18'), -- BAD: unknown customer
        (7,  1, 175.00, DATE '2024-01-16', DATE '2024-01-11'),  -- BAD: shipped before ordered
        (8,  2, 425.00, DATE '2024-01-17', DATE '2024-01-19'),  -- good
        (9,  3, 310.00, DATE '2024-01-18', DATE '2024-01-20'),  -- good
        (10, 4, 115.00, DATE '2024-01-19', DATE '2024-01-21')   -- good
""")

conn.close()

print(f"Database: {db_path}")

# Verify
c = duckdb.connect(db_path, read_only=True)
print("Customers:", c.execute("SELECT COUNT(*) FROM customers").fetchone()[0])
print("Orders:   ", c.execute("SELECT COUNT(*) FROM orders").fetchone()[0])
c.close()


### What just happened?
- **10 orders** were inserted: 7 valid, 2 with `shipped_date < order_date`, and 1 with a non-existent `customer_id = 999`.
- These three bad rows are exactly what each check today will catch: logic violations, referential integrity, and custom metrics.
- Using a temp `.duckdb` file means every Colab run starts clean — no stale data from previous runs.


## Step 2 · Failed Rows Check: shipped_date Before order_date

The `failed rows` check type lets you embed **arbitrary SQL** that returns the rows violating a business rule. Soda counts those rows and compares the count (or percentage) to thresholds.

```yaml
checks for orders:
  - failed rows:
      name: shipped_before_ordered
      fail condition: shipped_date < order_date
      fail: when > 0
```

The `fail condition` is a SQL `WHERE` clause fragment — Soda wraps it in `SELECT COUNT(*) FROM table WHERE <condition>`. No JOIN or subquery needed for single-table checks.

> **Production equivalent:** Replace `fail condition` with a full SQL query (`fail query:`) when the check requires JOINs or CTEs.


In [ ]:
from soda.scan import Scan

# Write Soda configuration
config_yml = f"""
data_sources:
  shopdb:
    type: duckdb
    path: "{db_path}"
"""
config_path = tmpdir / "configuration.yml"
config_path.write_text(config_yml)

# Failed-rows check: shipped_date must not precede order_date
failed_rows_yml = """
checks for orders:
  - failed rows:
      name: shipped_before_ordered
      fail condition: shipped_date < order_date
      fail: when > 0
"""

fr_path = tmpdir / "failed_rows.yml"
fr_path.write_text(failed_rows_yml)

scan = Scan()
scan.set_data_source_name("shopdb")
scan.add_configuration_yaml_file(str(config_path))
scan.add_sodacl_yaml_file(str(fr_path))
scan.execute()

print(scan.get_logs_text())
for check in scan.get_checks():
    print(f"[{check.outcome.upper()}] {check.name}")

# Manually verify which rows are the offenders
c = duckdb.connect(db_path, read_only=True)
bad = c.execute("""
    SELECT id, order_date, shipped_date
    FROM orders
    WHERE shipped_date < order_date
""").fetchall()
c.close()
print(f"\nOffending rows ({len(bad)}): {bad}")


### What just happened?
- Soda found **2 rows** where `shipped_date < order_date` and the check **failed** (threshold was `> 0`).
- The `fail condition` is pure SQL — any expression valid in a `WHERE` clause works, including functions, casts, and date arithmetic.
- The manual DuckDB query at the end confirms which row IDs are the problem — useful for debugging before writing the fix.
- **No schema changes** are needed: the check reads the table as-is.


## Step 3 · Custom Metric: revenue_per_order

Soda lets you define **user-defined metrics** using SQL expressions. A custom metric computes a single scalar value across the table, then applies standard threshold checks.

```yaml
checks for orders:
  - revenue_per_order between 50 and 500:
      revenue_per_order expression: SUM(revenue) / COUNT(*)
```

The `expression:` block tells Soda how to compute the metric. Soda runs:
```sql
SELECT SUM(revenue) / COUNT(*) FROM orders
```
and then checks whether the result satisfies `between 50 and 500`.

| Metric expression | Example use case |
|---|---|
| `SUM(col) / COUNT(*)` | Average revenue per order |
| `COUNT(DISTINCT col)` | Unique customers this period |
| `SUM(CASE WHEN status='error' THEN 1 ELSE 0 END)` | Error row count |


In [ ]:
# Custom metric: average revenue per order must be between $50 and $500
custom_metric_yml = """
checks for orders:
  - revenue_per_order between 50 and 500:
      name: avg_revenue_per_order_in_range
      revenue_per_order expression: SUM(revenue) / COUNT(*)
"""

cm_path = tmpdir / "custom_metric.yml"
cm_path.write_text(custom_metric_yml)

scan2 = Scan()
scan2.set_data_source_name("shopdb")
scan2.add_configuration_yaml_file(str(config_path))
scan2.add_sodacl_yaml_file(str(cm_path))
scan2.execute()

print(scan2.get_logs_text())
for check in scan2.get_checks():
    print(f"[{check.outcome.upper()}] {check.name}")

# Compute the metric manually to verify
c = duckdb.connect(db_path, read_only=True)
result = c.execute("SELECT ROUND(SUM(revenue) / COUNT(*), 2) FROM orders").fetchone()[0]
c.close()
print(f"\nManual check: SUM(revenue)/COUNT(*) = ${result}")
print(f"Expected range: $50 – $500")
print(f"Outcome: {'PASS' if 50 <= result <= 500 else 'FAIL'}")


### What just happened?
- Soda evaluated `SUM(revenue) / COUNT(*)` across all 10 orders and compared the result to the `[50, 500]` range.
- The average is ~$236 — well within range — so the check **passed**.
- **Custom metrics decouple business logic from schema**: if `revenue` is renamed, you update only the expression, not the check name.
- The manual DuckDB verification confirms the number Soda computed internally.


## Step 4 · Referential Integrity: Every order.customer_id Must Exist in customers

Referential integrity checks require a JOIN across two tables. Soda supports this via the `failed rows query:` syntax, which accepts a full SQL statement:

```yaml
checks for orders:
  - failed rows:
      name: orphan_customer_ids
      fail query: |
        SELECT o.id, o.customer_id
        FROM orders o
        LEFT JOIN customers c ON o.customer_id = c.id
        WHERE c.id IS NULL
      fail: when > 0
```

Soda executes the query and counts the returned rows. Each returned row is a **failed record** — an order whose `customer_id` has no matching entry in `customers`.

> **Production tip:** Multi-table integrity checks are the main reason to use `fail query:` over `fail condition:`. The `fail condition:` form only supports single-table `WHERE` clauses.


In [ ]:
# Referential integrity: every order must reference a valid customer
ref_integrity_yml = """
checks for orders:
  - failed rows:
      name: orphan_customer_ids
      fail query: |
        SELECT o.id, o.customer_id
        FROM orders o
        LEFT JOIN customers c ON o.customer_id = c.id
        WHERE c.id IS NULL
      fail: when > 0
"""

ri_path = tmpdir / "ref_integrity.yml"
ri_path.write_text(ref_integrity_yml)

scan3 = Scan()
scan3.set_data_source_name("shopdb")
scan3.add_configuration_yaml_file(str(config_path))
scan3.add_sodacl_yaml_file(str(ri_path))
scan3.execute()

print(scan3.get_logs_text())
for check in scan3.get_checks():
    print(f"[{check.outcome.upper()}] {check.name}")

# Show the orphan rows directly
c = duckdb.connect(db_path, read_only=True)
orphans = c.execute("""
    SELECT o.id AS order_id, o.customer_id
    FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.id
    WHERE c.id IS NULL
""").fetchall()
c.close()
print(f"\nOrphan orders ({len(orphans)}): {orphans}")


### What just happened?
- Order #6 references `customer_id = 999`, which does not exist in the `customers` table — the check **failed**.
- **`fail query:`** runs the entire SQL block; Soda counts the rows returned (not a scalar metric).
- This pattern generalises to any cross-table business rule: product IDs in a fact table must exist in a dimension table, invoice lines must reference valid invoice headers, etc.
- The orphan query at the end gives the exact row ID to audit, making remediation fast.


## Step 5 · Warn/Fail Percentage Thresholds on Failed Rows

Hard `fail: when > 0` thresholds are strict. For noisy real-world data you often want:
- **WARN** at a small percentage (e.g. 0.1% of rows) — alert the team
- **FAIL** at a larger percentage (e.g. 1% of rows) — block the pipeline

SodaCL supports percentage thresholds with the `%` suffix:

```yaml
checks for orders:
  - failed rows:
      name: shipped_before_ordered_pct
      fail condition: shipped_date < order_date
      warn: when > 0.1%
      fail: when > 1%
```

The percentage is computed as `failed_rows / total_rows * 100`. With 10 rows and 2 failures, the rate is 20% — above both thresholds.


In [ ]:
# Combined failed-rows check with warn + fail percentage thresholds
pct_checks_yml = """
checks for orders:
  - failed rows:
      name: shipped_before_ordered_pct
      fail condition: shipped_date < order_date
      warn: when > 0.1%
      fail: when > 1%
"""

pct_path = tmpdir / "pct_checks.yml"
pct_path.write_text(pct_checks_yml)

scan4 = Scan()
scan4.set_data_source_name("shopdb")
scan4.add_configuration_yaml_file(str(config_path))
scan4.add_sodacl_yaml_file(str(pct_path))
scan4.execute()

print(scan4.get_logs_text())
for check in scan4.get_checks():
    print(f"[{check.outcome.upper()}] {check.name}")

# Show the actual failure percentage
c = duckdb.connect(db_path, read_only=True)
total = c.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
failed = c.execute(
    "SELECT COUNT(*) FROM orders WHERE shipped_date < order_date"
).fetchone()[0]
c.close()
pct = failed / total * 100
print(f"\nFailed rows: {failed}/{total} = {pct:.1f}%")
print(f"Warn threshold: 0.1%  |  Fail threshold: 1%")
print(f"→ {pct:.1f}% is above both thresholds → FAIL")


### What just happened?
- 2 out of 10 rows fail the condition → **20%** failure rate, which exceeds the 1% fail threshold.
- When both `warn` and `fail` are exceeded, Soda reports the **highest severity** (fail takes precedence over warn).
- **Percentage thresholds** make checks resilient to table growth — a fixed `fail: when > 5` becomes irrelevant on a 10M-row table, but `fail: when > 1%` stays meaningful.
- In the real world the 0.1%/1% split is common: warn triggers an investigation, fail blocks the pipeline.


## Step 6 · Combining All Custom Check Types in One Scan

A real data quality scan runs all checks together. Combining them in a single YAML file means:
- One connection open/close cycle (more efficient)
- A single exit code for the whole batch
- One structured result list for routing to alerting systems


In [ ]:
# All custom checks in one scan
all_checks_yml = """
checks for orders:
  # 1. Shipped-date logic violation (hard fail)
  - failed rows:
      name: shipped_before_ordered
      fail condition: shipped_date < order_date
      fail: when > 0

  # 2. Custom metric: average revenue must be in a sensible range
  - revenue_per_order between 50 and 500:
      name: avg_revenue_per_order
      revenue_per_order expression: SUM(revenue) / COUNT(*)

  # 3. Referential integrity via full query
  - failed rows:
      name: orphan_customer_ids
      fail query: |
        SELECT o.id, o.customer_id
        FROM orders o
        LEFT JOIN customers c ON o.customer_id = c.id
        WHERE c.id IS NULL
      fail: when > 0

  # 4. Percentage-based warn/fail for shipped-date violations
  - failed rows:
      name: shipped_before_ordered_pct_gate
      fail condition: shipped_date < order_date
      warn: when > 0.1%
      fail: when > 1%
"""

all_path = tmpdir / "all_checks.yml"
all_path.write_text(all_checks_yml)

scan5 = Scan()
scan5.set_data_source_name("shopdb")
scan5.add_configuration_yaml_file(str(config_path))
scan5.add_sodacl_yaml_file(str(all_path))
scan5.execute()

# Structured summary table
print("=" * 55)
print(f"{'CHECK NAME':<35} {'OUTCOME':>8}")
print("=" * 55)
for check in scan5.get_checks():
    icon = {"pass": "✓", "warn": "⚠", "fail": "✗"}.get(check.outcome, "?")
    print(f"  {icon} {check.name:<33} {check.outcome.upper():>8}")

print()
print(f"Exit code: {scan5.get_exit_code()}  |  Any failures: {scan5.has_check_failures()}")


### What just happened?
- All four custom check types ran in a **single scan** with one connection open/close.
- The custom metric `avg_revenue_per_order` **passed** (~$237 in range).
- Both `shipped_before_ordered` checks and `orphan_customer_ids` **failed** due to the intentional bad data.
- **Exit code 2** confirms at least one failure — this is the signal to raise in a pipeline operator.
- The structured summary table is easy to feed into a Slack webhook or a metadata store.


In [ ]:
# Challenge: New table — products with referential integrity
#
# 1. In the same ecommerce.duckdb, create a table `order_lines`:
#      id INTEGER, order_id INTEGER, product_id INTEGER, quantity INTEGER
# 2. Insert 8 rows:
#    - 6 rows with valid order_id values (from orders.id: 1–10)
#    - 2 rows with order_id = 999 (no matching order)
#    - 1 row with quantity = -5 (invalid)
# 3. Write a SodaCL checks file for `order_lines` that:
#    a. Uses fail_condition to check quantity >= 1 (hard fail > 0)
#    b. Uses fail_query for referential integrity: order_id must exist in orders
#       warn: when > 5%, fail: when > 15%
# 4. Run the scan and print each check name + outcome
#
# Hint: you will need a new configuration.yml pointing to the same db_path
# and the checks should target `order_lines` (not `orders`)

# Your solution here
# conn2 = duckdb.connect(db_path)
# conn2.execute("CREATE TABLE order_lines (...)")
# ...


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `fail condition:` | SQL `WHERE` fragment for single-table failed-rows checks |
| `fail query:` | Full SQL statement for multi-table checks (JOINs, CTEs) |
| Custom metric | `metric_name expression: SQL_expr` — computes a scalar, then applies thresholds |
| Referential integrity | LEFT JOIN + `WHERE c.id IS NULL` finds orphan rows |
| Percentage thresholds | `warn: when > 0.1%` / `fail: when > 1%` — resilient to table growth |
| Severity precedence | When both warn and fail thresholds are exceeded, `fail` wins |

> **Tip:** Use `fail condition:` for single-table logic checks and `fail query:` the moment you need a JOIN — trying to encode referential integrity as a `WHERE` condition without joining will silently pass every time.

---
## What's next
**Day 6** → Integrating Soda scans into Airflow, Prefect, and CI/CD — turn today's checks into first-class pipeline gates.

Mark Day 5 complete in your [tracker](../index.html).
